In [0]:
# Read CSV directly from GitHub using pandas, convert to Spark
import pandas as pd

url = "https://raw.githubusercontent.com/julianahubacova/smb-loan-quality-pipeline/main/data/sample/loan_applications.csv"

# Read with pandas first
pdf = pd.read_csv(url)
print(f"Loaded {len(pdf)} rows from GitHub")
pdf.head()

In [0]:
# ── BRONZE LAYER ──────────────────────────────────────────────
# Raw ingestion - no transformations, just land the data as-is
# with metadata columns added for traceability

from pyspark.sql.functions import current_timestamp, lit

# Convert pandas dataframe to Spark dataframe
df_bronze = spark.createDataFrame(pdf)

# Add metadata columns (standard practice in real pipelines)
df_bronze = df_bronze \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", lit("loan_applications.csv")) \
    .withColumn("_layer", lit("bronze"))

# Save as Delta table
df_bronze.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_loan_applications")

print(f"Bronze layer loaded: {df_bronze.count()} rows")
df_bronze.printSchema()

In [0]:
# ── SILVER LAYER ──────────────────────────────────────────────
# Cleaned, typed, and deduplicated data
# This is the "trusted" layer

from pyspark.sql.functions import col, to_date, trim, upper
from pyspark.sql.types import IntegerType, DecimalType

# Read from Bronze
df_silver = spark.table("bronze_loan_applications")

# 1. Fix data types
df_silver = df_silver \
    .withColumn("application_date", to_date(col("application_date"), "yyyy-MM-dd")) \
    .withColumn("loan_amount", col("loan_amount").cast(DecimalType(15, 2))) \
    .withColumn("credit_score", col("credit_score").cast(IntegerType())) \
    .withColumn("employee_count", col("employee_count").cast(IntegerType())) \
    .withColumn("years_in_business", col("years_in_business").cast(IntegerType()))

# 2. Standardize text fields
df_silver = df_silver \
    .withColumn("province", trim(upper(col("province")))) \
    .withColumn("status", trim(col("status")))

# 3. Remove duplicates
df_silver = df_silver.dropDuplicates(["application_id"])

# 4. Update layer tag
df_silver = df_silver \
    .withColumn("_layer", lit("silver"))

# Save as Delta table
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_loan_applications")

print(f"Silver layer loaded: {df_silver.count()} rows")
df_silver.printSchema()

In [0]:
# ── GOLD LAYER ──────────────────────────────────────────────
# Aggregated, business-ready data
# This is what analysts and dashboards consume

from pyspark.sql.functions import count, sum, avg, round, when

# Read from Silver
df_gold_base = spark.table("silver_loan_applications")

# 1. Loans by province
df_gold_province = df_gold_base.groupBy("province") \
    .agg(
        count("application_id").alias("total_applications"),
        sum(when(col("status") == "Approved", 1).otherwise(0)).alias("approved"),
        sum(when(col("status") == "Rejected", 1).otherwise(0)).alias("rejected"),
        sum(when(col("status") == "Under Review", 1).otherwise(0)).alias("under_review"),
        round(avg("loan_amount"), 2).alias("avg_loan_amount"),
        round(sum("loan_amount"), 2).alias("total_loan_volume")
    ) \
    .withColumn("_layer", lit("gold"))

# 2. Loans by industry sector
df_gold_industry = df_gold_base.groupBy("industry_sector") \
    .agg(
        count("application_id").alias("total_applications"),
        round(avg("loan_amount"), 2).alias("avg_loan_amount"),
        round(avg("credit_score"), 0).alias("avg_credit_score")
    ) \
    .withColumn("_layer", lit("gold"))

# Save both as Delta tables
df_gold_province.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_loans_by_province")

df_gold_industry.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_loans_by_industry")

print(f"Gold layer loaded!")
print(f"\n Loans by Province:")
df_gold_province.show()
print(f"\n Loans by Industry:")
df_gold_industry.show()